Outer maximization : By dankin's theorem, we just need to ensure that the delta is maximum. In training, it means that the delta is detached during training


We are using pgd as the inner maximization approximation and we are using L infinity as the threat model 



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset



In [ ]:
def _bcast(v, like):
    """Reshape a per-example vector of shape (B,) so it broadcasts against a (B, ...) tensor."""
    return v.view(-1, *([1] * (like.dim() - 1)))


def _per_example_norm(t, p):
    """Norm of each example in the batch, flattening everything but the batch dim. Shape (B,)."""
    return t.flatten(1).norm(p=p, dim=1)

In [ ]:
def project(delta, eps, norm):
    """Euclidean projection of each example's delta onto {||delta||_norm <= eps}."""
    if norm == "linf":
        return delta.clamp(-eps, eps)
    if norm == "l2":
        # scale = 1 if already inside the ball, else eps / ||delta|| (pull back onto the sphere)
        scale = (eps / _per_example_norm(delta, 2).clamp_min(1e-12)).clamp(max=1.0)
        return delta * _bcast(scale, delta)
    raise ValueError(f"unknown norm: {norm}")


def ascent_direction(grad, norm):
    """
    Steepest-ascent direction under the given norm, i.e. argmax_{||v|| <= 1} <grad, v>.
      l_inf -> sign(grad)            (every coordinate moves by the full step)
      l_2   -> grad / ||grad||_2     (normalize PER EXAMPLE, not over the whole batch)
    """
    if norm == "linf":
        return grad.sign()
    if norm == "l2":
        return grad / _bcast(_per_example_norm(grad, 2).clamp_min(1e-12), grad)
    raise ValueError(f"unknown norm: {norm}")

In [ ]:
def random_init(x, eps, norm):
    """Random start inside the eps-ball. Helps when the gradient at delta = 0 is uninformative."""
    if norm == "linf":
        return torch.empty_like(x).uniform_(-eps, eps)
    if norm == "l2":
        d = torch.randn_like(x)
        d = d / _bcast(_per_example_norm(d, 2), d)                  # random unit direction
        r = torch.rand(x.shape[0], device=x.device) * eps          # random radius in [0, eps]
        return d * _bcast(r, d)
    raise ValueError(f"unknown norm: {norm}")


def clip_to_domain(x, delta, clip_min, clip_max):
    """Keep x + delta a valid input (e.g. pixels in [0, 1]). Pass None for both to skip."""
    if clip_min is None and clip_max is None:
        return delta
    return (x + delta).clamp(min=clip_min, max=clip_max) - x


Now we will write code for the **Inner Maximization** via *PGD* attack


In [ ]:
def pgd_attack(model, x, eps, alpha, steps, norm='linf',random_start = True, restarts = 1, clip_min= 0.0, clip_max = 1.0):
    was_training = model.training
    model.eval()

    best_delta = torch.zeros_like(x)
    best_loss = torch.full((x.shape[0],), -float("inf"), device=x.device)

    # calculate the best delta "restart" times
    for _ in range ( restarts ):
        delta = random_init(x, eps, norm) if random_start == True else torch.zeros_like(x)
        delta = clip_to_domain(x, delta, clip_min=clip_min, clip_max=clip_max)

        # delta maximization loop 
        for _ in range (steps):

            #calculating the grad wrt delta
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y, reduction="sum")
            (grad, ) = torch.autograd.grad(loss, delta)

            # actual delta calculation
            with torch.no_grad():
                delta = delta + alpha * ascent_direction(grad, norm)  # 1) ascent step (PLUS sign!)
                delta = project(delta, eps, norm)                     # 2) back into Δ
                delta = clip_to_domain(x, delta, clip_min, clip_max)  # 3) valid input range

        with torch.no_grad():
            per_ex_loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = per_ex_loss > best_loss
            best_delta[better] = delta[better]
            best_loss = torch.where(better, per_ex_loss, best_loss)

        model.train(was_training)
        return best_delta.detach()


